# HMC for gauge theory

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # use GPU 2

import torch
import numpy as np
from matplotlib import pyplot as plt

from lattice_ml.monte_carlo import SUnHMD, SUnHMC

from lattice_ml.gauge_tools import WilsonGaugeAction

from normflow.prior import SUnPrior

# HMD

First we see how the action and lop-probabilty of the injected momena change with HMD steps.

In [ ]:
lat_shape = (16, 16)

prior = SUnPrior(n=3, shape=(*lat_shape, len(lat_shape)))
action = WilsonGaugeAction(beta=4)

hmd = SUnHMD(lambda t, q: action.algebra_force(q), t_span=(0, 1), num_steps=10)


x = prior.sample(4)

y, delta_log_prop_momentum = hmd.step(x)

action(y) - action(x), delta_log_prop_momentum

(tensor([-1109.2372, -1223.0605, -1185.9997, -1178.7941], device='cuda:0'),
 tensor([-1099.2450, -1209.6814, -1172.6848, -1166.5010], device='cuda:0'))

# HMC
Now we perform HMC simulations

In [ ]:
beta = 6
lat_shape = (16, 16)

prior = SUnPrior(n=3, shape=(*lat_shape, len(lat_shape)))

action = WilsonGaugeAction(beta=beta)

hmc = SUnHMC(lambda t, q: action.algebra_force(q), t_span=(0, 1), num_steps=15, action=action)

In [4]:
num_samples = 1024
x = prior.sample(num_samples)

num_links = np.prod(lat_shape) * 1


def analyize(x):
    plaq_mean = action(x).mean() / (-beta * num_links)
    plaq_error = action(x).std() / (beta * num_links) / num_samples ** 0.5
    return f"{plaq_mean.item():.4f}({10_000 * plaq_error.item():0.0f})"

import time

t_0 = time.time()

for k in range(200):
    x, is_accepted = hmc.step(x)
    if k % 10 == 0:
        print(f"{k}\t{torch.sum(is_accepted).item() / num_samples:.2f}\t", analyize(x))


print(f"Total time: {time.time() - t_0}")

0	1.00	 1.4324(13)
10	0.90	 3.2524(22)
20	0.79	 3.4777(20)
30	0.72	 3.5514(18)
40	0.70	 3.5742(16)
50	0.69	 3.5812(15)
60	0.67	 3.5814(16)
70	0.71	 3.5823(15)
80	0.68	 3.5822(15)
90	0.69	 3.5802(15)
100	0.67	 3.5810(15)
110	0.69	 3.5805(15)
120	0.69	 3.5802(15)
130	0.70	 3.5791(15)
140	0.71	 3.5816(16)
150	0.68	 3.5789(16)
160	0.69	 3.5810(15)
170	0.71	 3.5795(15)
180	0.70	 3.5816(16)
190	0.68	 3.5797(16)
Total time: 1589.7322916984558


In [ ]:
torch.save(x, "/scratch/lturgut/cfgs1024_b6_2dim_200traj_15steps.pt")